In [0]:
# path Edu
pagamento = spark.read.parquet("/Volumes/workspace/hackathon_2025/default/source/book_pagamento/dados_pagamento/")

In [0]:
# path Michael
pagamento = spark.read.parquet("/Volumes/hackathon_2025/default/source/book_pagamento/dados_pagamento/")

In [0]:
display(pagamento)

In [0]:
pagamento.createOrReplaceTempView("pagamento")

In [0]:
%sql
SELECT * from pagamento
WHERE NUM_CPF = "79X9TWTZ8TW"
ORDER BY "DAT_STATUS_FATURA"

# 20260117 Michael

In [0]:
%sql
-- Volumetria total + range temporal (datas principais)
SELECT
  COUNT(*) AS total_linhas,
  MIN(to_timestamp(DAT_STATUS_FATURA,'ddMMMyyyy:HH:mm:ss')) AS min_dat_status_fatura,
  MAX(to_timestamp(DAT_STATUS_FATURA,'ddMMMyyyy:HH:mm:ss')) AS max_dat_status_fatura,
  MIN(to_timestamp(DAT_STATUS_PAGAMENTO,'ddMMMyyyy:HH:mm:ss')) AS min_dat_status_pagamento,
  MAX(to_timestamp(DAT_STATUS_PAGAMENTO,'ddMMMyyyy:HH:mm:ss')) AS max_dat_status_pagamento
FROM pagamento;

In [0]:
%sql
-- Parsing de datas (taxa de inválidos) 
SELECT
  COUNT(*) AS total,

  SUM(CASE WHEN DAT_STATUS_FATURA IS NULL OR TRIM(DAT_STATUS_FATURA)='' THEN 1 ELSE 0 END) AS null_dat_status_fatura,
  SUM(CASE WHEN DAT_STATUS_FATURA IS NOT NULL AND TRIM(DAT_STATUS_FATURA)<>'' 
            AND to_timestamp(DAT_STATUS_FATURA,'ddMMMyyyy:HH:mm:ss') IS NULL THEN 1 ELSE 0 END) AS invalid_dat_status_fatura,

  SUM(CASE WHEN DAT_STATUS_PAGAMENTO IS NULL OR TRIM(DAT_STATUS_PAGAMENTO)='' THEN 1 ELSE 0 END) AS null_dat_status_pagamento,
  SUM(CASE WHEN DAT_STATUS_PAGAMENTO IS NOT NULL AND TRIM(DAT_STATUS_PAGAMENTO)<>'' 
            AND to_timestamp(DAT_STATUS_PAGAMENTO,'ddMMMyyyy:HH:mm:ss') IS NULL THEN 1 ELSE 0 END) AS invalid_dat_status_pagamento
FROM pagamento;

In [0]:
%sql
-- Grão e duplicidade: qual é a “linha única”
-- Teste A (Fatura)

SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT CONCAT(CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA)) AS distinct_fatura_item,
  COUNT(*) - COUNT(DISTINCT CONCAT(CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA)) AS duplicadas_fatura_item
FROM pagamento;

In [0]:
%sql
-- Grão e duplicidade: qual é a “linha única”
-- Teste B (Pagamentos)

SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT CONCAT(SEQ_ENTIDADE_PAGAMENTO,'#',NUM_FATURA_PAGAMENTO,'#',COD_FORMA_PAGAMENTO,'#',VAL_ATUAL_PAGAMENTO)) AS distinct_pagamento,
  COUNT(*) - COUNT(DISTINCT CONCAT(SEQ_ENTIDADE_PAGAMENTO,'#',NUM_FATURA_PAGAMENTO,'#',COD_FORMA_PAGAMENTO,'#',VAL_ATUAL_PAGAMENTO)) AS duplicadas_pagamento
FROM pagamento;

In [0]:
%sql
-- Domínios (DISTINCT) das colunas de status/tipo mais importantes
SELECT DISTINCT IND_STATUS_FATURA FROM pagamento;

In [0]:
%sql
SELECT DISTINCT IND_STATUS_PAGAMENTO FROM pagamento;

In [0]:
%sql
SELECT DISTINCT COD_METODO_PAGAMENTO FROM pagamento;

In [0]:
%sql
SELECT DISTINCT DW_TIPO_FATURA FROM pagamento;

In [0]:
%sql
SELECT DISTINCT COD_TIPO_FATURA FROM pagamento;

In [0]:
%sql
SELECT DISTINCT COD_FORMA_PAGAMENTO FROM pagamento;

In [0]:
%sql
SELECT DISTINCT IND_TIPO_CREDITO FROM pagamento;

In [0]:
%sql
SELECT DISTINCT COD_TIPO_PAGAMENTO FROM pagamento;

In [0]:
%sql
SELECT DISTINCT DW_TIPO_PAGAMENTO  FROM pagamento;

In [0]:
%sql
-- Valores monetários: negativos, zeros, outliers (mínimo viável)

In [0]:
%sql
SELECT
  SUM(CASE WHEN CAST(VAL_PAGAMENTO_FATURA AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS neg_pag_fatura,
  SUM(CASE WHEN CAST(VAL_PAGAMENTO_ITEM AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS neg_pag_item,
  SUM(CASE WHEN CAST(VAL_ATUAL_PAGAMENTO AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS neg_val_atual_pag,
  SUM(CASE WHEN CAST(VAL_PAGAMENTO_CREDITO AS DOUBLE) < 0 THEN 1 ELSE 0 END) AS neg_pag_credito,
  SUM(CASE WHEN CAST(VAL_JUROS_MULTAS_ITEM AS DOUBLE) = 0 THEN 1 ELSE 0 END) AS neg_juros_multas_item,
  SUM(CASE WHEN CAST(VAL_ORIGINAL_PAGAMENTO AS DOUBLE) = 0 THEN 1 ELSE 0 END) AS neg_original_pagamento
FROM pagamento;

In [0]:
%sql
SELECT
  MIN(CAST(VAL_PAGAMENTO_FATURA AS DOUBLE)) AS min_val_atual,
  MAX(CAST(VAL_PAGAMENTO_FATURA AS DOUBLE)) AS max_val_atual,
  AVG(CAST(VAL_PAGAMENTO_FATURA AS DOUBLE)) AS avg_val_atual,
  percentile_approx(CAST(VAL_PAGAMENTO_FATURA AS DOUBLE), array(0.01,0.05,0.50,0.95,0.99), 10000) AS pctl_val_pg_fatura
FROM pagamento
WHERE VAL_PAGAMENTO_FATURA IS NOT NULL AND TRIM(VAL_PAGAMENTO_FATURA)<>'' AND CAST(VAL_PAGAMENTO_FATURA AS DOUBLE) IS NOT NULL;

In [0]:
%sql
-- Duplicidade no nível fatura-item (candidato mais natural)
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT CONCAT(CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ)) AS distinct_fatura_item,
  COUNT(*) - COUNT(DISTINCT CONCAT(CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ)) AS duplicadas_fatura_item
FROM pagamento;

In [0]:
%sql
-- Duplicidade no nível fatura-item (candidato mais natural)
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT CONCAT(CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ,'#', NUM_CPF)) AS distinct_fatura_item,
  COUNT(*) - COUNT(DISTINCT CONCAT(CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ,'#', NUM_CPF)) AS duplicadas_fatura_item
FROM pagamento;

In [0]:
%sql
-- Duplicidade no nível pagamento (entidade de pagamento)
-- CONTRATO + SEQ_FATURA + NUM_SUB_SEQ_FATURA + NUM_CREDITO_SEQ + NUM_CPF

SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT CONCAT(SEQ_ENTIDADE_PAGAMENTO,'#',NUM_FATURA_PAGAMENTO,'#',COD_TIPO_PAGAMENTO,'#',VAL_ATUAL_PAGAMENTO)) AS distinct_pagamento,
  COUNT(*) - COUNT(DISTINCT CONCAT(SEQ_ENTIDADE_PAGAMENTO,'#',NUM_FATURA_PAGAMENTO,'#',COD_TIPO_PAGAMENTO,'#',VAL_ATUAL_PAGAMENTO)) AS duplicadas_pagamento
FROM pagamento;

In [0]:
%sql
-- Quantas chaves duplicadas existem? (e qual a maior repetição)
WITH k AS (
  SELECT
    CONCAT(NUM_CPF,'#',CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ) AS DEDUP_KEY,
    COUNT(*) AS qtd
  FROM pagamento
  GROUP BY 1
)
SELECT
  SUM(CASE WHEN qtd > 1 THEN 1 ELSE 0 END) AS num_chaves_duplicadas,
  SUM(CASE WHEN qtd > 1 THEN qtd ELSE 0 END) AS total_linhas_em_chaves_duplicadas,
  MAX(qtd) AS max_repeticao
FROM k;

In [0]:
%sql
--- “TS_STATUS_FATURA DESC” deixa empates? (mesma chave, mesmo TS)
--- Essa query mede, dentro das chaves duplicadas, se existe mais de uma linha com o mesmo TS_STATUS_FATURA máximo — ou seja, o ordenamento por TS_STATUS_FATURA DESC não define um vencedor único.Essa query mede, dentro das chaves duplicadas, se existe mais de uma linha com o mesmo TS_STATUS_FATURA máximo — ou seja, o ordenamento por TS_STATUS_FATURA DESC não define um vencedor único.

WITH base AS (
  SELECT
    CONCAT(NUM_CPF,'#',CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ) AS DEDUP_KEY,
    to_timestamp(upper(DAT_STATUS_FATURA),'ddMMMyyyy:HH:mm:ss') AS TS_STATUS_FATURA
  FROM pagamento
),
max_ts AS (
  SELECT
    DEDUP_KEY,
    MAX(TS_STATUS_FATURA) AS TS_MAX
  FROM base
  GROUP BY 1
),
tied AS (
  SELECT
    b.DEDUP_KEY,
    COUNT(*) AS qtd_no_ts_max
  FROM base b
  JOIN max_ts m
    ON b.DEDUP_KEY = m.DEDUP_KEY
   AND b.TS_STATUS_FATURA = m.TS_MAX
  GROUP BY 1
)
SELECT
  SUM(CASE WHEN qtd_no_ts_max > 1 THEN 1 ELSE 0 END) AS chaves_com_empate_no_ts_max,
  MAX(qtd_no_ts_max) AS maior_empate_observado
FROM tied;

In [0]:
%sql
WITH typed AS (
  SELECT
    *,
    to_timestamp(upper(DAT_STATUS_FATURA),'ddMMMyyyy:HH:mm:ss') AS TS_STATUS_FATURA,
    CONCAT(NUM_CPF,'#',CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ) AS DEDUP_KEY
  FROM pagamento
),
ranked AS (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY DEDUP_KEY
      ORDER BY TS_STATUS_FATURA DESC
    ) AS rn
  FROM typed
)
SELECT
  COUNT(*) AS linhas_pos_dedup
FROM ranked
WHERE rn = 1;

In [0]:
%sql
create or replace temp view pagamentos_dedup AS
WITH typed AS (
  SELECT
    *,
    to_timestamp(upper(DAT_STATUS_FATURA),'ddMMMyyyy:HH:mm:ss') AS TS_STATUS_FATURA,
    CONCAT(NUM_CPF,'#',CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ) AS DEDUP_KEY
  FROM pagamento
),
ranked AS (
  SELECT
    *,
    row_number() OVER (
      PARTITION BY DEDUP_KEY
      ORDER BY TS_STATUS_FATURA DESC
    ) AS rn
  FROM typed
)
SELECT * EXCEPT (rn)
FROM ranked
WHERE rn = 1;

In [0]:
%sql
SELECT * FROM pagamentos_dedup

In [0]:
%sql
WITH typed AS (
  SELECT
    to_timestamp(upper(DAT_STATUS_FATURA),'ddMMMyyyy:HH:mm:ss') AS TS_STATUS_FATURA,
    CONCAT(NUM_CPF,'#',CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ) AS DEDUP_KEY
  FROM pagamento
),
ranked AS (
  SELECT
    row_number() OVER (
      PARTITION BY DEDUP_KEY
      ORDER BY TS_STATUS_FATURA DESC
    ) AS rn
  FROM typed
)
SELECT
  COUNT(*) AS total_in,
  SUM(CASE WHEN rn = 1 THEN 1 ELSE 0 END) AS total_out,
  COUNT(*) - SUM(CASE WHEN rn = 1 THEN 1 ELSE 0 END) AS linhas_removidas
FROM ranked;

In [0]:
%sql
-- listar as chaves duplicadas

WITH base AS (
  SELECT
    CONCAT(NUM_CPF,'#',CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ) AS DEDUP_KEY
  FROM pagamento
),
dups AS (
  SELECT DEDUP_KEY
  FROM base
  GROUP BY 1
  HAVING COUNT(*) > 1
)
SELECT COUNT(*) AS qtd_chaves_duplicadas
FROM dups;

In [0]:
%sql
-- Ver as diferenças entre as duas linhas (campos de “estado”)

WITH typed AS (
  SELECT
    CONCAT(NUM_CPF,'#',CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ) AS DEDUP_KEY,
    NUM_CPF, CONTRATO, SEQ_FATURA, NUM_SUB_SEQ_FATURA, NUM_CREDITO_SEQ,

    DAT_STATUS_FATURA,
    to_timestamp(upper(DAT_STATUS_FATURA),'ddMMMyyyy:HH:mm:ss') AS TS_STATUS_FATURA,

    IND_STATUS_FATURA,
    IND_STATUS_PAGAMENTO,
    DAT_STATUS_PAGAMENTO,
    to_timestamp(upper(DAT_STATUS_PAGAMENTO),'ddMMMyyyy:HH:mm:ss') AS TS_STATUS_PAGAMENTO,

    VAL_PAGAMENTO_FATURA,
    VAL_PAGAMENTO_ITEM,
    VAL_ATUAL_PAGAMENTO,
    VAL_ORIGINAL_PAGAMENTO,
    VAL_JUROS_MULTAS_ITEM,
    VAL_DESCONTO_ITEM,

    SEQ_ENTIDADE_PAGAMENTO,
    SEQ_ENTIDADE_ATIVIDADE,
    SEQ_ENTIDADE_CREDITO
  FROM pagamento
),
dups AS (
  SELECT DEDUP_KEY
  FROM typed
  GROUP BY 1
  HAVING COUNT(*) > 1
),
ranked AS (
  SELECT
    *,
    row_number() OVER (PARTITION BY DEDUP_KEY ORDER BY TS_STATUS_FATURA DESC) AS rn
  FROM typed
)
SELECT
  DEDUP_KEY,
  rn,
  DAT_STATUS_FATURA, TS_STATUS_FATURA,
  IND_STATUS_FATURA,
  IND_STATUS_PAGAMENTO, DAT_STATUS_PAGAMENTO, TS_STATUS_PAGAMENTO,
  VAL_PAGAMENTO_FATURA, VAL_PAGAMENTO_ITEM, VAL_ATUAL_PAGAMENTO, VAL_ORIGINAL_PAGAMENTO,
  VAL_JUROS_MULTAS_ITEM, VAL_DESCONTO_ITEM,
  SEQ_ENTIDADE_PAGAMENTO, SEQ_ENTIDADE_ATIVIDADE, SEQ_ENTIDADE_CREDITO
FROM ranked
WHERE DEDUP_KEY IN (SELECT DEDUP_KEY FROM dups)
ORDER BY DEDUP_KEY, rn
LIMIT 20;

In [0]:
%sql
-- Dentro das chaves duplicadas, quantas vezes mudam campos-chave entre rn=1 e rn=2.

WITH typed AS (
  SELECT
    CONCAT(NUM_CPF,'#',CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ) AS DEDUP_KEY,
    to_timestamp(upper(DAT_STATUS_FATURA),'ddMMMyyyy:HH:mm:ss') AS TS_STATUS_FATURA,

    IND_STATUS_FATURA,
    IND_STATUS_PAGAMENTO,
    to_timestamp(upper(DAT_STATUS_PAGAMENTO),'ddMMMyyyy:HH:mm:ss') AS TS_STATUS_PAGAMENTO,

    CAST(NULLIF(TRIM(VAL_ATUAL_PAGAMENTO),'') AS DOUBLE) AS VAL_ATUAL_PAGAMENTO,
    CAST(NULLIF(TRIM(VAL_JUROS_MULTAS_ITEM),'') AS DOUBLE) AS VAL_JUROS_MULTAS_ITEM,

    SEQ_ENTIDADE_PAGAMENTO
  FROM pagamento
),
ranked AS (
  SELECT
    *,
    row_number() OVER (PARTITION BY DEDUP_KEY ORDER BY TS_STATUS_FATURA DESC) AS rn
  FROM typed
),
pairs AS (
  SELECT
    a.DEDUP_KEY,

    a.IND_STATUS_FATURA AS ind_fatura_new,
    b.IND_STATUS_FATURA AS ind_fatura_old,

    a.IND_STATUS_PAGAMENTO AS ind_pag_new,
    b.IND_STATUS_PAGAMENTO AS ind_pag_old,

    a.VAL_ATUAL_PAGAMENTO AS val_new,
    b.VAL_ATUAL_PAGAMENTO AS val_old,

    a.VAL_JUROS_MULTAS_ITEM AS juros_new,
    b.VAL_JUROS_MULTAS_ITEM AS juros_old,

    a.SEQ_ENTIDADE_PAGAMENTO AS seq_pag_new,
    b.SEQ_ENTIDADE_PAGAMENTO AS seq_pag_old
  FROM ranked a
  JOIN ranked b
    ON a.DEDUP_KEY = b.DEDUP_KEY
   AND a.rn = 1
   AND b.rn = 2
)
SELECT
  SUM(CASE WHEN ind_fatura_new <> ind_fatura_old THEN 1 ELSE 0 END) AS dif_status_fatura,
  SUM(CASE WHEN coalesce(ind_pag_new,'') <> coalesce(ind_pag_old,'') THEN 1 ELSE 0 END) AS dif_status_pagamento,
  SUM(CASE WHEN val_new <> val_old THEN 1 ELSE 0 END) AS dif_val_atual_pag,
  SUM(CASE WHEN juros_new <> juros_old THEN 1 ELSE 0 END) AS dif_juros,
  SUM(CASE WHEN coalesce(seq_pag_new,'') <> coalesce(seq_pag_old,'') THEN 1 ELSE 0 END) AS dif_seq_entidade_pagamento,
  COUNT(*) AS total_pares
FROM pairs;

In [0]:
%sql
-- padrão de mudança em IND_STATUS_PAGAMENTO

WITH typed AS (
  SELECT
    CONCAT(NUM_CPF,'#',CONTRATO,'#',SEQ_FATURA,'#',NUM_SUB_SEQ_FATURA,'#',NUM_CREDITO_SEQ) AS DEDUP_KEY,
    to_timestamp(upper(DAT_STATUS_FATURA),'ddMMMyyyy:HH:mm:ss') AS TS_STATUS_FATURA,
    IND_STATUS_PAGAMENTO
  FROM pagamento
),
ranked AS (
  SELECT
    *,
    row_number() OVER (PARTITION BY DEDUP_KEY ORDER BY TS_STATUS_FATURA DESC) AS rn
  FROM typed
),
pairs AS (
  SELECT
    a.IND_STATUS_PAGAMENTO AS status_new,
    b.IND_STATUS_PAGAMENTO AS status_old
  FROM ranked a
  JOIN ranked b
    ON a.DEDUP_KEY = b.DEDUP_KEY
   AND a.rn = 1
   AND b.rn = 2
)
SELECT
  coalesce(status_old,'NULL') AS status_old,
  coalesce(status_new,'NULL') AS status_new,
  COUNT(*) AS qtd
FROM pairs
GROUP BY 1,2
ORDER BY qtd DESC
LIMIT 20;